In [ ]:
import pandas as pd
import numpy as np
import unicodedata, os

DATA_DIR = "/content"

def read(name):
    return pd.read_csv(os.path.join(DATA_DIR, name))

customers   = read("olist_customers_dataset.csv")
geolocation = read("olist_geolocation_dataset.csv")
items       = read("olist_order_items_dataset.csv")
payments    = read("olist_order_payments_dataset.csv")
reviews     = read("olist_order_reviews_dataset.csv")
orders      = read("olist_orders_dataset.csv")
products    = read("olist_products_dataset.csv")
sellers     = read("olist_sellers_dataset.csv")
translation = read("product_category_name_translation.csv")

tables = {"customers": customers, "geolocation": geolocation, "items": items,
          "payments": payments, "reviews": reviews, "orders": orders,
          "products": products, "sellers": sellers, "translation": translation}

for name, df in tables.items():
    print(f"{name:12s} {df.shape[0]:>10,} rows  {df.shape[1]} columns")

customers        99,441 rows  5 columns
geolocation   1,000,163 rows  5 columns
items           112,650 rows  7 columns
payments        103,886 rows  5 columns
reviews          99,224 rows  7 columns
orders           99,441 rows  8 columns
products         32,951 rows  9 columns
sellers           3,095 rows  4 columns
translation          71 rows  2 columns


In [ ]:
summary = pd.DataFrame({
    "rows": {k: len(v) for k, v in tables.items()},
    "duplicate_rows": {k: int(v.duplicated().sum()) for k, v in tables.items()},
})
display(summary)

for name, df in tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f"\n{name}:")
        print(nulls)

,rows,duplicate_rows
customers,99441,0
geolocation,1000163,261831
items,112650,0
payments,103886,0
reviews,99224,0
orders,99441,0
products,32951,0
sellers,3095,0
translation,71,0



reviews:
review_comment_title      87656
review_comment_message    58247
dtype: int64

orders:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

products:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


In [ ]:
orders.info()
display(orders.head())
print(orders["order_status"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [ ]:
log = []

def log_step(table, issue, action, rows_affected):
    log.append({"table": table, "issue": issue, "action": action, "rows_affected": rows_affected})
    print(f"{table}: {issue} -> {action} ({rows_affected:,} rows)")

In [ ]:
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]

for c in date_cols:
    orders[c] = pd.to_datetime(orders[c])

log_step("orders", "dates stored as text", "converted 5 columns to datetime", len(orders))

print(orders["order_purchase_timestamp"].min(), "to", orders["order_purchase_timestamp"].max())

monthly = orders["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()
print(monthly.to_string())

orders: dates stored as text -> converted 5 columns to datetime (99,441 rows)
2016-09-04 21:15:19 to 2018-10-17 17:30:18
order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M


In [ ]:
check = orders.assign(
    no_approval=orders["order_approved_at"].isna(),
    no_carrier=orders["order_delivered_carrier_date"].isna(),
    no_delivery=orders["order_delivered_customer_date"].isna(),
)
print(check.groupby("order_status")[["no_approval", "no_carrier", "no_delivery"]].sum())

              no_approval  no_carrier  no_delivery
order_status                                      
approved                0           2            2
canceled              141         550          619
created                 5           5            5
delivered              14           2            8
invoiced                0         314          314
processing              0         301          301
shipped                 0           0         1107
unavailable             0         609          609


In [ ]:
orders["order_date"] = orders["order_purchase_timestamp"].dt.normalize()

orders["in_scope"] = orders["order_date"].between("2017-01-01", "2018-08-31")
log_step("orders", "sparse months outside Jan 2017 - Aug 2018", "flagged in_scope = False", int((~orders["in_scope"]).sum()))

orders["is_excluded"] = orders["order_status"].isin(["canceled", "unavailable"])
log_step("orders", "canceled / unavailable orders", "flagged is_excluded = True (not counted as revenue)", int(orders["is_excluded"].sum()))

delivered_ok = (orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].notna()
orders["missing_delivery_date"] = (orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna()
log_step("orders", "delivered orders with no delivery date", "flagged missing_delivery_date = True", int(orders["missing_delivery_date"].sum()))

orders["delivery_days"] = ((orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"])
                           .dt.total_seconds() / 86400).where(delivered_ok).round(1)

orders: sparse months outside Jan 2017 - Aug 2018 -> flagged in_scope = False (349 rows)
orders: canceled / unavailable orders -> flagged is_excluded = True (not counted as revenue) (1,234 rows)
orders: delivered orders with no delivery date -> flagged missing_delivery_date = True (8 rows)


In [ ]:
print("delivery before purchase:", int((orders["delivery_days"] < 0).sum()))
print("delivered before carrier pickup:", int((orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum()))
print("canceled orders that have a delivery date:", int(((orders["order_status"] == "canceled") & orders["order_delivered_customer_date"].notna()).sum()))
print()
print(orders["in_scope"].value_counts())
print()
print(orders["delivery_days"].describe().round(1))

delivery before purchase: 0
delivered before carrier pickup: 23
canceled orders that have a delivery date: 6

in_scope
True     99092
False      349
Name: count, dtype: int64

count    96470.0
mean        12.6
std          9.5
min          0.5
25%          6.8
50%         10.2
75%         15.7
max        209.6
Name: delivery_days, dtype: float64


In [ ]:
for days in [30, 60, 90, 120]:
    print(f"delivery over {days} days:", int((orders["delivery_days"] > days).sum()))

log = [e for e in log if e["issue"] not in ("delivered before carrier pickup date",
                                            "extreme delivery times (over 60 days)")]

orders["carrier_date_after_delivery"] = orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]
log_step("orders", "delivered before carrier pickup date", "flagged carrier_date_after_delivery = True (kept)", int(orders["carrier_date_after_delivery"].sum()))

orders["extreme_delivery"] = orders["delivery_days"] > 60
log_step("orders", "extreme delivery times (over 60 days)", "flagged extreme_delivery = True (kept)", int(orders["extreme_delivery"].sum()))

mean_all = round(orders["delivery_days"].mean(), 1)
median_all = round(orders["delivery_days"].median(), 1)
mean_60 = round(orders.loc[orders["delivery_days"] <= 60, "delivery_days"].mean(), 1)

print()
print("mean:", mean_all, "| median:", median_all)
print("mean without orders over 60 days:", mean_60)

delivery over 30 days: 4524
delivery over 60 days: 304
delivery over 90 days: 77
delivery over 120 days: 43
orders: delivered before carrier pickup date -> flagged carrier_date_after_delivery = True (kept) (23 rows)
orders: extreme delivery times (over 60 days) -> flagged extreme_delivery = True (kept) (304 rows)

mean: 12.6 | median: 10.2
mean without orders over 60 days: 12.3


In [ ]:
print("rows:", len(customers))
print("unique customer_id:", customers["customer_id"].nunique())
print("unique customer_unique_id:", customers["customer_unique_id"].nunique())
print()
print(customers["customer_state"].value_counts().head(10))
print()
odd = customers["customer_city"].str.contains(r"[^a-z ]", regex=True)
print("cities with capitals, accents or symbols:", int(odd.sum()))
print(customers.loc[odd, "customer_city"].value_counts().head(10))

rows: 99441
unique customer_id: 99441
unique customer_unique_id: 96096

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: int64

cities with capitals, accents or symbols: 451
customer_city
santa barbara d'oeste    119
mogi-guacu                93
embu-guacu                46
dias d'avila              24
ji-parana                 17
pariquera-acu             15
mirassol d'oeste          14
biritiba-mirim            10
herval d'oeste             9
arraial d'ajuda            8
Name: count, dtype: int64


In [ ]:
print("states:", customers["customer_state"].nunique())
print("cities:", customers["customer_city"].nunique())
print("cities with leading/trailing spaces:", int((customers["customer_city"] != customers["customer_city"].str.strip()).sum()))
print()
print("zip prefix type:", customers["customer_zip_code_prefix"].dtype)
print("zip prefixes with fewer than 5 digits:", int((customers["customer_zip_code_prefix"] < 10000).sum()))
print()
orders_per_customer = customers.groupby("customer_unique_id").size()
print("customers with more than one order:", int((orders_per_customer > 1).sum()))
print(orders_per_customer.value_counts().sort_index().head(6))

states: 27
cities: 4119
cities with leading/trailing spaces: 0

zip prefix type: int64
zip prefixes with fewer than 5 digits: 23995

customers with more than one order: 2997
1    93099
2     2745
3      203
4       30
5        8
6        6
Name: count, dtype: int64


In [ ]:
affected = 0
for df, col in [(customers, "customer_zip_code_prefix"),
                (sellers, "seller_zip_code_prefix"),
                (geolocation, "geolocation_zip_code_prefix")]:
    affected += int((pd.to_numeric(df[col]) < 10000).sum())
    df[col] = df[col].astype(str).str.zfill(5)

log_step("customers, sellers, geolocation", "ZIP prefix stored as number (leading zeros lost)",
         "converted to 5-character text", affected)
log_step("customers", "city and state text", "checked, already consistent, no change needed", 0)

# checks
for df, col in [(customers, "customer_zip_code_prefix"),
                (sellers, "seller_zip_code_prefix"),
                (geolocation, "geolocation_zip_code_prefix")]:
    print(col, "| all 5 characters:", bool((df[col].str.len() == 5).all()))

print()
zips_in_geo = set(geolocation["geolocation_zip_code_prefix"])
print("customers whose ZIP is not in geolocation:", int((~customers["customer_zip_code_prefix"].isin(zips_in_geo)).sum()))
print("sellers whose ZIP is not in geolocation:", int((~sellers["seller_zip_code_prefix"].isin(zips_in_geo)).sum()))

customers, sellers, geolocation: ZIP prefix stored as number (leading zeros lost) -> converted to 5-character text (270,755 rows)
customers: city and state text -> checked, already consistent, no change needed (0 rows)
customer_zip_code_prefix | all 5 characters: True
seller_zip_code_prefix | all 5 characters: True
geolocation_zip_code_prefix | all 5 characters: True

customers whose ZIP is not in geolocation: 278
sellers whose ZIP is not in geolocation: 7


In [ ]:
before = len(geolocation)
geolocation = geolocation.drop_duplicates()
log_step("geolocation", "exact duplicate rows", "dropped", before - len(geolocation))

print("rows now:", len(geolocation))
print("distinct ZIP prefixes:", geolocation["geolocation_zip_code_prefix"].nunique())
print()
print(geolocation.groupby("geolocation_zip_code_prefix").size().describe().round(1))

# Brazil sits roughly between lat -34 and 6, lng -74 and -34
outside = ~geolocation["geolocation_lat"].between(-34, 6) | ~geolocation["geolocation_lng"].between(-74, -34)
print()
print("coordinates outside Brazil:", int(outside.sum()))
print(geolocation[outside].head())

print()
odd = geolocation["geolocation_city"].str.contains(r"[^a-z ]", regex=True)
print("city names with accents, capitals or symbols:", int(odd.sum()))
print(geolocation.loc[odd, "geolocation_city"].value_counts().head(10))

geolocation: exact duplicate rows -> dropped (261,831 rows)
rows now: 738332
distinct ZIP prefixes: 19015

count    19015.0
mean        38.8
std         50.9
min          1.0
25%          8.0
50%         23.0
75%         49.0
max        779.0
dtype: float64

coordinates outside Brazil: 33
       geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
387565                       18243        28.008978       -15.536867   
513631                       28165        41.614052        -8.411675   
513643                       28155       -34.586422       -58.732101   
513754                       28155        42.439286        13.820214   
514429                       28333        38.381672        -6.328200   

               geolocation_city geolocation_state  
387565  bom retiro da esperanca                SP  
513631      vila nova de campos                RJ  
513643              santa maria                RJ  
513754              santa maria                RJ  
514429           

In [ ]:
def norm_text(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode("ascii")
    return s.strip().lower()

zips_before = geolocation["geolocation_zip_code_prefix"].nunique()

# 1. drop coordinates outside Brazil
before = len(geolocation)
outside = ~geolocation["geolocation_lat"].between(-34, 6) | ~geolocation["geolocation_lng"].between(-74, -34)
geolocation = geolocation[~outside].copy()
log_step("geolocation", "coordinates outside Brazil", "dropped rows", before - len(geolocation))

# 2. standardize city names
old_city = geolocation["geolocation_city"].copy()
geolocation["geolocation_city"] = geolocation["geolocation_city"].map(norm_text)
log_step("geolocation", "city names with accents", "converted to lowercase without accents",
         int((old_city != geolocation["geolocation_city"]).sum()))

# 3. one row per ZIP prefix
geo_zip = (geolocation.groupby("geolocation_zip_code_prefix")
           .agg(lat=("geolocation_lat", "median"),
                lng=("geolocation_lng", "median"),
                city=("geolocation_city", lambda s: s.mode().iloc[0]),
                state=("geolocation_state", lambda s: s.mode().iloc[0]))
           .reset_index())
log_step("geolocation", "many coordinate rows per ZIP prefix (up to 779)",
         "collapsed to one row per ZIP (median coordinates, most common city and state)",
         len(geolocation) - len(geo_zip))

print()
print("rows:", len(geo_zip))
print("ZIP prefixes before:", zips_before, "| after:", geo_zip["geolocation_zip_code_prefix"].nunique())
print("ZIP unique:", geo_zip["geolocation_zip_code_prefix"].is_unique)
print("lat range:", round(geo_zip["lat"].min(), 1), "to", round(geo_zip["lat"].max(), 1))
print("lng range:", round(geo_zip["lng"].min(), 1), "to", round(geo_zip["lng"].max(), 1))
geo_zip.head()

geolocation: coordinates outside Brazil -> dropped rows (33 rows)
geolocation: city names with accents -> converted to lowercase without accents (63,434 rows)
geolocation: many coordinate rows per ZIP prefix (up to 779) -> collapsed to one row per ZIP (median coordinates, most common city and state) (719,289 rows)

rows: 19010
ZIP prefixes before: 19015 | after: 19010
ZIP unique: True
lat range: -33.7 to 4.5
lng range: -72.9 to -34.8


,geolocation_zip_code_prefix,lat,lng,city,state
0,01001,-23.549951,-46.634027,sao paulo,SP
1,01002,-23.548228,-46.635247,sao paulo,SP
2,01003,-23.548977,-46.635313,sao paulo,SP
3,01004,-23.549550,-46.634771,sao paulo,SP
4,01005,-23.549763,-46.636100,sao paulo,SP


In [ ]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
log_step("order_items", "shipping limit date stored as text", "converted to datetime", len(items))

items["line_total"] = items["price"] + items["freight_value"]

print(items[["price", "freight_value", "line_total"]].describe().round(2))
print()
print("duplicate (order_id, order_item_id):", int(items.duplicated(["order_id", "order_item_id"]).sum()))
print("price <= 0:", int((items["price"] <= 0).sum()))
print("freight < 0:", int((items["freight_value"] < 0).sum()))
print("freight = 0:", int((items["freight_value"] == 0).sum()))
print()
print("items whose order is missing from orders:", int((~items["order_id"].isin(orders["order_id"])).sum()))
print("items whose product is missing from products:", int((~items["product_id"].isin(products["product_id"])).sum()))
print("items whose seller is missing from sellers:", int((~items["seller_id"].isin(sellers["seller_id"])).sum()))
print()
no_items = ~orders["order_id"].isin(items["order_id"])
print("orders with no items:", int(no_items.sum()))
print(orders.loc[no_items, "order_status"].value_counts())


order_items: shipping limit date stored as text -> converted to datetime (112,650 rows)
           price  freight_value  line_total
count  112650.00      112650.00   112650.00
mean      120.65          19.99      140.64
std       183.63          15.81      190.72
min         0.85           0.00        6.08
25%        39.90          13.08       55.22
50%        74.99          16.26       92.32
75%       134.90          21.15      157.94
max      6735.00         409.68     6929.31

duplicate (order_id, order_item_id): 0
price <= 0: 0
freight < 0: 0
freight = 0: 383

items whose order is missing from orders: 0
items whose product is missing from products: 0
items whose seller is missing from sellers: 0

orders with no items: 775
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [ ]:
orders["has_items"] = orders["order_id"].isin(items["order_id"])
log_step("orders", "orders with no items (mostly canceled / unavailable)",
         "flagged has_items = False (no revenue)", int((~orders["has_items"]).sum()))

top = (items.sort_values("price", ascending=False).head(8)
       .merge(products[["product_id", "product_category_name"]], on="product_id", how="left"))
print(top[["order_id", "price", "freight_value", "product_category_name"]])

print()
print("items per order:")
print(items.groupby("order_id").size().value_counts().sort_index().head(8))

orders: orders with no items (mostly canceled / unavailable) -> flagged has_items = False (no revenue) (775 rows)
                           order_id    price  freight_value  \
0  0812eb902a67711a1cb742b3cdaa65ae  6735.00         194.31   
1  fefacc66af859508bf1a7934eab1e97f  6729.00         193.21   
2  f5136e38d1a14a4dbd87dff67da82701  6499.00         227.66   
3  a96610ab360d42a2e5335a3998b4718a  4799.00         151.34   
4  199af31afc78c699f0dbf71fb178d4d4  4690.00          74.34   
5  8dbc85d1447242f3b127dda390d56e19  4590.00          91.78   
6  426a9742b533fc6fed17d1fd6d143d7e  4399.87         113.45   
7  68101694e5c5dc7330c91e1bbc36214f  4099.99          75.27   

   product_category_name  
0  utilidades_domesticas  
1                    pcs  
2                  artes  
3        eletroportateis  
4        eletroportateis  
5                    pcs  
6  instrumentos_musicais  
7         consoles_games  

items per order:
1    88863
2     7516
3     1322
4      505
5      204
6 

In [ ]:
print(payments["payment_type"].value_counts())
print()
print(payments[["payment_installments", "payment_value"]].describe().round(2))
print()
zero = payments["payment_value"] == 0
print("zero-value payments:", int(zero.sum()))
print(payments.loc[zero, "payment_type"].value_counts())
print()
print("installments = 0:", int((payments["payment_installments"] == 0).sum()))
print("orders with more than one payment row:", int((payments.groupby("order_id").size() > 1).sum()))
print("orders with no payment:", int((~orders["order_id"].isin(payments["order_id"])).sum()))

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

       payment_installments  payment_value
count             103886.00      103886.00
mean                   2.85         154.10
std                    2.69         217.49
min                    0.00           0.00
25%                    1.00          56.79
50%                    1.00         100.00
75%                    4.00         171.84
max                   24.00       13664.08

zero-value payments: 9
payment_type
voucher        6
not_defined    3
Name: count, dtype: int64

installments = 0: 2
orders with more than one payment row: 2961
orders with no payment: 1


In [ ]:
payments["is_zero_payment"] = payments["payment_value"] == 0
payments["invalid_installments"] = payments["payment_installments"] == 0
log_step("payments", "zero-value payments (voucher and not_defined)", "flagged is_zero_payment = True (kept)", int(payments["is_zero_payment"].sum()))
log_step("payments", "installments = 0", "flagged invalid_installments = True (kept)", int(payments["invalid_installments"].sum()))

# one row per order
pay_order = payments.groupby("order_id").agg(
    payment_total=("payment_value", "sum"),
    payment_rows=("payment_value", "size"),
    max_installments=("payment_installments", "max"),
).reset_index()

main_type = (payments.sort_values("payment_value", ascending=False)
             .drop_duplicates("order_id")[["order_id", "payment_type"]]
             .rename(columns={"payment_type": "main_payment_type"}))
pay_order = pay_order.merge(main_type, on="order_id")

# compare payments with price + freight
item_totals = items.groupby("order_id")["line_total"].sum().rename("items_total").reset_index()
recon = pay_order.merge(item_totals, on="order_id", how="inner")
recon["diff"] = (recon["payment_total"] - recon["items_total"]).round(2)
mismatch = recon[recon["diff"].abs() > 1]

print("orders compared:", len(recon))
print("payments differ from items by more than 1:", len(mismatch))
print("payment higher:", int((mismatch["diff"] > 0).sum()), "| payment lower:", int((mismatch["diff"] < 0).sum()))
print()
print(mismatch["diff"].describe().round(2))
print()
print(mismatch["main_payment_type"].value_counts())
print("average installments, mismatched orders:", round(mismatch["max_installments"].mean(), 1),
      "| all orders:", round(recon["max_installments"].mean(), 1))
print()
print(orders.loc[~orders["order_id"].isin(payments["order_id"]), ["order_id", "order_status"]])

payments: zero-value payments (voucher and not_defined) -> flagged is_zero_payment = True (kept) (9 rows)
payments: installments = 0 -> flagged invalid_installments = True (kept) (2 rows)
orders compared: 98665
payments differ from items by more than 1: 249
payment higher: 232 | payment lower: 17

count    249.00
mean      11.51
std       19.35
min      -51.62
25%        3.29
50%        6.98
75%       14.12
max      182.81
Name: diff, dtype: float64

main_payment_type
credit_card    239
debit_card       7
voucher          3
Name: count, dtype: int64
average installments, mismatched orders: 6.3 | all orders: 2.9

                               order_id order_status
30710  bfbd0f9bdef84302105ad712db648a6c    delivered


In [ ]:
pay_order = pay_order.merge(recon[["order_id", "items_total", "diff"]], on="order_id", how="left")
pay_order["payment_mismatch"] = pay_order["diff"].abs() > 1
log_step("payments", "payment total differs from items price + freight by more than 1",
         "flagged payment_mismatch = True (revenue is based on item price, not payments)",
         int(pay_order["payment_mismatch"].sum()))

orders["has_payment"] = orders["order_id"].isin(payments["order_id"])
log_step("orders", "order with no payment record", "flagged has_payment = False (kept)",
         int((~orders["has_payment"]).sum()))

payments: payment total differs from items price + freight by more than 1 -> flagged payment_mismatch = True (revenue is based on item price, not payments) (249 rows)
orders: order with no payment record -> flagged has_payment = False (kept) (1 rows)


In [ ]:
for c in ["review_creation_date", "review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c])
log_step("reviews", "dates stored as text", "converted to datetime", len(reviews))

print(reviews["review_score"].value_counts().sort_index())
print()
per_order = reviews.groupby("order_id").size()
print("duplicate review_id:", int(reviews["review_id"].duplicated().sum()))
print("orders with more than one review:", int((per_order > 1).sum()))
print(per_order.value_counts().sort_index())
print()
print("reviews whose order is missing from orders:", int((~reviews["order_id"].isin(orders["order_id"])).sum()))
print("orders with no review:", int((~orders["order_id"].isin(reviews["order_id"])).sum()))
print("answered before created:", int((reviews["review_answer_timestamp"] < reviews["review_creation_date"]).sum()))
print()
sample = per_order[per_order > 1].index[:3]
print(reviews[reviews["order_id"].isin(sample)].sort_values(["order_id", "review_answer_timestamp"])
      [["order_id", "review_id", "review_score", "review_creation_date", "review_answer_timestamp"]])

reviews: dates stored as text -> converted to datetime (99,224 rows)
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

duplicate review_id: 814
orders with more than one review: 547
1    98126
2      543
3        4
Name: count, dtype: int64

reviews whose order is missing from orders: 0
orders with no review: 768
answered before created: 0

                               order_id                         review_id  \
22423  0035246a40f520710769010f752e7507  2a74b0559eb58fc1ff842ecc999594cb   
25612  0035246a40f520710769010f752e7507  89a02c45c340aeeb1354a24e7d4b2c1e   
22779  013056cfe49763c6f66bda03396c5ee3  ab30810c29da5da8045216f0f62652a2   
68633  013056cfe49763c6f66bda03396c5ee3  73413b847f63e02bc752b364f6d05ee9   
854    0176a6846bcb3b0d3aa3116a9a768597  830636803620cdf8b6ffaf1b2f6e92b2   
83224  0176a6846bcb3b0d3aa3116a9a768597  d8e8c42271c8fb67b9dad95d98c8ff80   

       review_score review_creation_date review_answer_timestamp  
22423

In [ ]:
before = len(reviews)
reviews = (reviews.sort_values(["order_id", "review_answer_timestamp"])
           .drop_duplicates("order_id", keep="last")
           .reset_index(drop=True))
log_step("reviews", "orders with more than one review (547 orders)",
         "kept the most recent review per order", before - len(reviews))

reviews["has_comment"] = reviews["review_comment_message"].notna()
log_step("reviews", "comment text is mostly blank", "added has_comment flag (comments not used in KPIs)",
         int((~reviews["has_comment"]).sum()))

print("rows:", len(reviews), "| one review per order:", reviews["order_id"].is_unique)
print("average score:", round(reviews["review_score"].mean(), 2))

reviews: orders with more than one review (547 orders) -> kept the most recent review per order (551 rows)
reviews: comment text is mostly blank -> added has_comment flag (comments not used in KPIs) (57,898 rows)
rows: 98673 | one review per order: True
average score: 4.09


In [ ]:
products = products.rename(columns={"product_name_lenght": "product_name_length",
                                    "product_description_lenght": "product_description_length"})
log_step("products", "misspelled column names ('lenght')", "renamed to 'length'", 0)

print("rows:", len(products), "| unique product_id:", products["product_id"].nunique())
print("categories:", products["product_category_name"].nunique())
print("products with no category:", int(products["product_category_name"].isna().sum()))
missing = sorted(set(products["product_category_name"].dropna()) - set(translation["product_category_name"]))
print("categories with no English translation:", missing)
print()
print(products[["product_weight_g", "product_length_cm", "product_height_cm",
                "product_width_cm", "product_photos_qty"]].describe().round(1))
print("weight = 0:", int((products["product_weight_g"] == 0).sum()))
print()
print("products never sold:", int((~products["product_id"].isin(items["product_id"])).sum()))

products: misspelled column names ('lenght') -> renamed to 'length' (0 rows)
rows: 32951 | unique product_id: 32951
categories: 73
products with no category: 610
categories with no English translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']

       product_weight_g  product_length_cm  product_height_cm  \
count           32949.0            32949.0            32949.0   
mean             2276.5               30.8               16.9   
std              4282.0               16.9               13.6   
min                 0.0                7.0                2.0   
25%               300.0               18.0                8.0   
50%               700.0               25.0               13.0   
75%              1900.0               38.0               21.0   
max             40425.0              105.0              105.0   

       product_width_cm  product_photos_qty  
count           32949.0             32341.0  
mean               23.2                 2.2  
std        

In [ ]:

extra = {"pc_gamer": "pc_gamer",
         "portateis_cozinha_e_preparadores_de_alimentos": "portable_kitchen_food_preparers"}
translation = pd.concat([translation,
                         pd.DataFrame({"product_category_name": list(extra),
                                       "product_category_name_english": list(extra.values())})],
                        ignore_index=True)
log_step("products", "2 categories missing from translation table", "added the English names manually", len(extra))

typos = {"costruction_tools_garden": "construction_tools_garden",
         "costruction_tools_tools": "construction_tools_tools",
         "fashio_female_clothing": "fashion_female_clothing",
         "home_confort": "home_comfort"}
translation["product_category_name_english"] = translation["product_category_name_english"].replace(typos)
log_step("products", "typos in English category names", "corrected spelling", len(typos))

rows_before = len(products)
products = products.merge(translation, on="product_category_name", how="left")
no_category = int(products["product_category_name"].isna().sum())
products["category"] = (products["product_category_name_english"].fillna("unknown")
                        .str.replace("_", " ").str.title())
products = products.drop(columns=["product_category_name_english"])
log_step("products", "products with no category", "labeled 'Unknown'", no_category)

products["invalid_weight"] = products["product_weight_g"] == 0
log_step("products", "weight = 0", "flagged invalid_weight = True (weight not used in KPIs)", int(products["invalid_weight"].sum()))

print("rows before:", rows_before, "| after:", len(products), "| product_id unique:", products["product_id"].is_unique)
print(products["category"].nunique(), "categories")
print(products["category"].value_counts().head(10))

products: 2 categories missing from translation table -> added the English names manually (2 rows)
products: typos in English category names -> corrected spelling (4 rows)
products: products with no category -> labeled 'Unknown' (610 rows)
products: weight = 0 -> flagged invalid_weight = True (weight not used in KPIs) (4 rows)
rows before: 32951 | after: 32951 | product_id unique: True
74 categories
category
Bed Bath Table           3029
Sports Leisure           2867
Furniture Decor          2657
Health Beauty            2444
Housewares               2335
Auto                     1900
Computers Accessories    1639
Toys                     1411
Watches Gifts            1329
Telephony                1134
Name: count, dtype: int64


In [ ]:
print("rows:", len(sellers), "| unique seller_id:", sellers["seller_id"].nunique())
print(sellers["seller_state"].value_counts().head(5))
print()
odd = sellers["seller_city"].str.contains(r"[^a-z ]", regex=True)
print("seller cities with capitals, accents or symbols:", int(odd.sum()))
print(sellers.loc[odd, "seller_city"].value_counts().head(10))
print()
print("sellers with no sales:", int((~sellers["seller_id"].isin(items["seller_id"])).sum()))

rows: 3095 | unique seller_id: 3095
seller_state
SP    1849
PR     349
MG     244
SC     190
RJ     171
Name: count, dtype: int64

seller cities with capitals, accents or symbols: 34
seller_city
santa barbara d'oeste                       4
sao paulo - sp                              3
santa barbara d´oeste                       2
lages - sc                                  1
são paulo                                  1
sao paulo / sao paulo                       1
novo hamburgo, rio grande do sul, brasil    1
04482255                                    1
cariacica / es                              1
sao miguel d'oeste                          1
Name: count, dtype: int64

sellers with no sales: 0


In [ ]:
import re

def clean_city(s):
    s = str(s).replace("´", "'").replace("`", "'")
    s = norm_text(s)
    s = re.split(r" - | / |,", s)[0].strip()
    return s

old_city = sellers["seller_city"].copy()
sellers["seller_city"] = sellers["seller_city"].map(clean_city)

# a ZIP code typed into the city field: take the city from the geolocation table instead
has_digits = sellers["seller_city"].str.contains(r"\d")
zip_city = geo_zip.set_index("geolocation_zip_code_prefix")["city"]
sellers.loc[has_digits, "seller_city"] = (sellers.loc[has_digits, "seller_zip_code_prefix"]
                                          .map(zip_city).fillna("unknown"))

changed = old_city != sellers["seller_city"]
log_step("sellers", "city field with accents, state suffixes, extra text or a ZIP code",
         "standardized to a lowercase city name (ZIP-only value replaced using the seller's ZIP)", int(changed.sum()))

print(pd.DataFrame({"before": old_city[changed], "after": sellers.loc[changed, "seller_city"]}).to_string())
print()
print("cities left with digits or symbols:", int(sellers["seller_city"].str.contains(r"[^a-z' -]").sum()))

sellers: city field with accents, state suffixes, extra text or a ZIP code -> standardized to a lowercase city name (ZIP-only value replaced using the seller's ZIP) (18 rows)
                                        before                  after
78                                  lages - sc                  lages
246                      sao paulo / sao paulo              sao paulo
360                                 são paulo              sao paulo
476                      santa barbara d´oeste  santa barbara d'oeste
517                                   04482255         rio de janeiro
551   novo hamburgo, rio grande do sul, brasil          novo hamburgo
622                             cariacica / es              cariacica
826                             sao paulo - sp              sao paulo
1004                                   sp / sp                     sp
1337                      mogi das cruzes / sp        mogi das cruzes
1580                            sao paulo - sp         

In [ ]:
bad = sellers["seller_city"].str.contains(r"[^a-z' -]") | (sellers["seller_city"].str.len() <= 2)
print(sellers.loc[bad, ["seller_city", "seller_state", "seller_zip_code_prefix"]].to_string())


                         seller_city seller_state seller_zip_code_prefix
237                     auriflama/sp           SP                  15350
869                           sbc/sp           SP                  09726
874   arraial d'ajuda (porto seguro)           BA                  45816
945            santo andre/sao paulo           SP                  09230
1004                              sp           SP                  03363
1117                              sp           SP                  12903
1159                  maua/sao paulo           SP                  09380
1246                              sp           SP                  04776
1346  rio de janeiro \rio de janeiro           RJ                  22050
1447         barbacena/ minas gerais           MG                  36200
1712                      pinhais/pr           PR                  83327
1889                              sp           SP                  16021
2258       vendas@creditparts.com.br           PR  

In [ ]:
old_city = sellers["seller_city"].copy()

sellers["seller_city"] = sellers["seller_city"].map(lambda s: re.split(r"/|\\|\(", s)[0].strip())

bad = sellers["seller_city"].str.len().le(3) | sellers["seller_city"].str.contains(r"[^a-z' -]")
sellers.loc[bad, "seller_city"] = (sellers.loc[bad, "seller_zip_code_prefix"]
                                   .map(zip_city).fillna("unknown"))

changed = old_city != sellers["seller_city"]
log_step("sellers", "city field with extra text, state codes, abbreviations or an email",
         "kept the city name only; non-city values replaced using the seller's ZIP", int(changed.sum()))

print(pd.DataFrame({"before": old_city[changed], "after": sellers.loc[changed, "seller_city"]}).to_string())
print()
left = sellers["seller_city"].str.contains(r"[^a-z' -]") | sellers["seller_city"].str.len().le(3)
print("cities still odd:", int(left.sum()))
print("rows with city 'unknown':", int((sellers["seller_city"] == "unknown").sum()))

sellers: city field with extra text, state codes, abbreviations or an email -> kept the city name only; non-city values replaced using the seller's ZIP (16 rows)
                              before                   after
237                     auriflama/sp               auriflama
869                           sbc/sp   sao bernardo do campo
874   arraial d'ajuda (porto seguro)         arraial d'ajuda
945            santo andre/sao paulo             santo andre
1004                              sp               sao paulo
1117                              sp       braganca paulista
1159                  maua/sao paulo                    maua
1246                              sp               sao paulo
1346  rio de janeiro \rio de janeiro          rio de janeiro
1447         barbacena/ minas gerais               barbacena
1712                      pinhais/pr                 pinhais
1814                             sbc   sao bernardo do campo
1889                              sp         

In [ ]:
print(sellers.loc[left, ["seller_city", "seller_state", "seller_zip_code_prefix"]].to_string())

     seller_city seller_state seller_zip_code_prefix
197          itu           SP                  13304
244          uba           MG                  36502
470          poa           SP                  08550
478          itu           SP                  13310
550          uba           MG                  36500
663          itu           SP                  13312
946          ipe           RS                  95240
1185         jau           SP                  17210
1204         uba           MG                  36500
1232         uba           MG                  36500
1266         jau           SP                  17201
1408         itu           SP                  13311
1435         uba           MG                  36504
1515         uba           MG                  36500
2648         jau           SP                  17209
2834         jau           SP                  17209


In [ ]:
checks = {
    "orders: 99,441 rows, order_id unique": len(orders) == 99441 and orders["order_id"].is_unique,
    "order items: 112,650 rows, all orders exist": len(items) == 112650 and items["order_id"].isin(orders["order_id"]).all(),
    "payments_by_order: one row per order": pay_order["order_id"].is_unique,
    "reviews: one review per order, scores 1-5": reviews["order_id"].is_unique and reviews["review_score"].between(1, 5).all(),
    "products: 32,951 rows, product_id unique": len(products) == 32951 and products["product_id"].is_unique,
    "products: no missing category label": products["category"].notna().all(),
    "sellers: seller_id unique, no missing city": sellers["seller_id"].is_unique and sellers["seller_city"].notna().all(),
    "geolocation: one row per ZIP": geo_zip["geolocation_zip_code_prefix"].is_unique,
    "order dates are datetime": str(orders["order_purchase_timestamp"].dtype).startswith("datetime"),
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)
assert all(checks.values()), "a check failed"

# headline numbers: in scope, canceled and unavailable orders excluded
valid = items.merge(orders[["order_id", "order_date", "in_scope", "is_excluded"]], on="order_id")
valid = valid[valid["in_scope"] & ~valid["is_excluded"]]

print()
print("Revenue (item price): R$ {:,.0f}".format(valid["price"].sum()))
print("Orders:               {:,}".format(valid["order_id"].nunique()))
print("Average order value:  R$ {:,.2f}".format(valid.groupby("order_id")["price"].sum().mean()))

jan_aug = valid[valid["order_date"].dt.month <= 8]
print()
print(jan_aug.groupby(jan_aug["order_date"].dt.year).agg(revenue=("price", "sum"), orders=("order_id", "nunique")).round(0))

PASS - orders: 99,441 rows, order_id unique
PASS - order items: 112,650 rows, all orders exist
PASS - payments_by_order: one row per order
PASS - reviews: one review per order, scores 1-5
PASS - products: 32,951 rows, product_id unique
PASS - products: no missing category label
PASS - sellers: seller_id unique, no missing city
PASS - geolocation: one row per ZIP
PASS - order dates are datetime

Revenue (item price): R$ 13,449,530
Orders:               97,905
Average order value:  R$ 137.37

              revenue  orders
order_date                   
2017        3080851.0   22562
2018        7341037.0   53530


In [ ]:
import shutil

OUT_DIR = os.path.join(DATA_DIR, "clean")
os.makedirs(OUT_DIR, exist_ok=True)

def prep(df):
    # True/False flags are saved as 1/0 so SQL Server can load them
    df = df.copy()
    for c in df.select_dtypes(include="bool").columns:
        df[c] = df[c].astype(int)
    return df

# comment text is not used in any KPI, so it is left out of the export
reviews_export = reviews.drop(columns=["review_comment_title", "review_comment_message"])

exports = {
    "orders_clean.csv": orders,
    "order_items_clean.csv": items,
    "payments_clean.csv": payments,
    "payments_by_order.csv": pay_order,
    "reviews_clean.csv": reviews_export,
    "products_clean.csv": products,
    "sellers_clean.csv": sellers,
    "customers_clean.csv": customers,
    "geolocation_zip.csv": geo_zip,
    "cleaning_log.csv": pd.DataFrame(log),
}
for name, df in exports.items():
    prep(df).to_csv(os.path.join(OUT_DIR, name), index=False)
    print(f"{name:26s} {len(df):>9,} rows")

display(pd.DataFrame(log))

shutil.make_archive("/content/olist_clean", "zip", OUT_DIR)
try:
    from google.colab import files
    files.download("/content/olist_clean.zip")
except ImportError:
    pass

orders_clean.csv              99,441 rows
order_items_clean.csv        112,650 rows
payments_clean.csv           103,886 rows
payments_by_order.csv         99,440 rows
reviews_clean.csv             98,673 rows
products_clean.csv            32,951 rows
sellers_clean.csv              3,095 rows
customers_clean.csv           99,441 rows
geolocation_zip.csv           19,010 rows
cleaning_log.csv                  28 rows


,table,issue,action,rows_affected
0,orders,dates stored as text,converted 5 columns to datetime,99441
1,orders,sparse months outside Jan 2017 - Aug 2018,flagged in_scope = False,349
2,orders,canceled / unavailable orders,flagged is_excluded = True (not counted as rev...,1234
3,orders,delivered orders with no delivery date,flagged missing_delivery_date = True,8
4,orders,delivered before carrier pickup date,flagged carrier_date_after_delivery = True (kept),23
5,orders,extreme delivery times (over 60 days),flagged extreme_delivery = True (kept),304
6,"customers, sellers, geolocation",ZIP prefix stored as number (leading zeros lost),converted to 5-character text,270755
7,customers,city and state text,"checked, already consistent, no change needed",0
8,geolocation,exact duplicate rows,dropped,261831
9,geolocation,coordinates outside Brazil,dropped rows,33


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>